<a href="https://colab.research.google.com/github/Ananyasaini06/Small-Object-Detection/blob/main/Small_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup on a T4 runtime

In [1]:
%pip install -q "ultralytics>=8.3,<9" "sahi>=0.11.20,<0.12" "gradio>=5,<6"
from pathlib import Path
import subprocess, sys, torch
from ultralytics import YOLO
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → T4 GPU'
repo = Path('/content/Small-Object-Detection')
if not repo.exists():
    subprocess.run(['git','clone','--branch','main','https://github.com/Ananyasaini06/Small-Object-Detection.git',str(repo)], check=True)
sys.path.insert(0, str(repo))
from torchvision.ops import nms
from datetime import datetime, timezone
def run_id():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
from comparison import benchmark
from demo import launch


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 2. Mount Drive and recover a trusted VisDrone checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')
project = drive_root / 'Small-Object-Detection'
project.mkdir(exist_ok=True)
# Set this to your own trusted VisDrone best.pt or last.pt if stored elsewhere.
manual = None
candidates = [Path(manual)] if manual else []
if manual and not candidates[0].is_file():
    raise FileNotFoundError(manual)
for folder in (project, drive_root/'P10_runs', Path('/content/P10_runs'), repo):
    for filename in ('best.pt', 'last.pt'):
        candidates.extend(sorted(folder.glob('**/final_baseline*/weights/'+filename), key=lambda p:p.stat().st_mtime, reverse=True))
        candidates.extend([folder/filename])
weights = next((p for p in candidates if p.is_file()), None)
if weights:
    detector = YOLO(str(weights))
    assert len(detector.names) == 10, 'Verify this is a trained VisDrone checkpoint'
print('Recovered:', weights)

## 3. Prepare VisDrone for training or evaluation

In [ ]:
from ultralytics.data.utils import check_det_dataset, img2label_paths
data = check_det_dataset('VisDrone.yaml')
val_dir = Path(data['val'])
label_dir = Path(img2label_paths([str(val_dir / 'example.jpg')])[0]).parent
assert val_dir.is_dir() and label_dir.is_dir()


## 4. Optional training if recovery found no weights

In [ ]:
TRAIN = False
RESUME = False
if RESUME:
    assert weights, 'Recover a trusted checkpoint in section 2 first.'
    last = weights.with_name('last.pt')
    assert last.is_file(), 'Select the interrupted run in section 2.'
    model = YOLO(str(last))
    assert model.ckpt.get('optimizer') and model.ckpt.get('epoch', -1) >= 0, 'This checkpoint is finalized and cannot resume.'
    model.train(resume=True, device=0)
    weights = Path(model.trainer.best)
elif TRAIN and weights is None:
    model = YOLO('yolo11n.pt')
    model.train(data='VisDrone.yaml', epochs=15, imgsz=640, batch=8, device=0,
                workers=2, project=str(project), name='final_baseline', seed=42, patience=0, save=True)
    weights = Path(model.trainer.best)
print('Inference checkpoint:', weights)


## 5. Reproduce the corrected 100-image evaluation

In [ ]:
RUN_EVALUATION = True
REPEAT_HISTORICAL_PROTOCOL = False
if RUN_EVALUATION:
    assert weights and weights.is_file(), 'Recover or train YOLO first.'
    evaluation_dir = project / 'evaluation' / run_id()
    display(benchmark(weights, val_dir, label_dir, evaluation_dir))
    if REPEAT_HISTORICAL_PROTOCOL:
        from evaluate import evaluate
        evaluate(str(weights), val_dir, label_dir, evaluation_dir)
    print('Saved to', evaluation_dir)


## 6. Launch the upload demo

In [ ]:
RUN_DEMO = True
if RUN_DEMO:
    assert weights and weights.is_file(), 'Recover or train YOLO first.'
    launch(weights)


## 7. Save a submission backup to Drive

In [ ]:
import zipfile
assert weights and weights.is_file(), 'Recover or train YOLO first.'
backup = project / ('submission-' + run_id() + '.zip')
with zipfile.ZipFile(backup, 'x') as z:
    for path in {weights, weights.with_name('best.pt'), weights.with_name('last.pt')}:
        if path.is_file():
            z.write(path, 'weights/' + path.name)
    if 'evaluation_dir' in globals():
        for path in evaluation_dir.glob('*'):
            if path.is_file():
                z.write(path, 'evaluation/' + path.name)
    z.writestr('requirements-runtime.txt', subprocess.check_output([sys.executable,'-m','pip','freeze']))
    z.writestr('repository-commit.txt', subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD']))
print(backup)
